# HELM: Hierarchical and Explicit Label Modeling — Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marjanstoimchev/HELM/blob/master/notebooks/colab_demo.ipynb)

Hierarchical multi-label image classification with Vision Transformers, GraphSAGE, and BYOL self-supervision.

## 1. Setup

In [ ]:
%cd /content
!rm -rf HELM
!git clone https://github.com/marjanstoimchev/HELM.git
%cd HELM

# Install core dependencies
!pip install -q -r requirements.txt

# Install PyTorch Geometric (needs special handling for CUDA version)
import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
!pip install -q torch_geometric
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+cu{CUDA_VERSION}.html 2>/dev/null || echo "PyG extensions installed via fallback"

print("Setup complete!")

In [ ]:
import os, sys, gc, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import lightning as L
from lightning import seed_everything
from omegaconf import DictConfig, OmegaConf
import matplotlib.pyplot as plt

sys.path.insert(0, ".")

from data.dataset_pipeline import DatasetPipeline
from data.hierarchy import create_edge_index
from datamodules.base_datamodule import BaseDataModule
from models.model import h_deit_base_embedding
from augmentations import Preprocess
from callbacks import ModelCheckpoint_, EarlyStopping_, RichProgressBar_
from utils.utils import Dotdict, calculate_metrics, predict
from trainers import get_trainer_class

plt.rcParams.update({
    "font.family": "serif", "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 100, "axes.titleweight": "bold",
})

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

Choose the **dataset**, **method variant**, and **training parameters** below.

| Method | Description |
|--------|-------------|
| `hmlc-ssl-graph-byol` | **Full HELM** — hierarchy + graph + BYOL + semi-supervised |
| `hmlc-sl-graph-byol` | Hierarchy + graph + BYOL, supervised |
| `hmlc-ssl-graph` | Semi-supervised + graph, no BYOL |
| `hmlc-ssl-byol` | Semi-supervised + BYOL, no graph |
| `hmlc-sl` | Hierarchical, supervised only |
| `mlc-sl` | Flat multi-label baseline |

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────
DATASET = "ucm"                          # ucm, aid, mlrsnet, dfc_15, chestxray8, ...
DATASET_CONFIG = f"configs/dataset/{DATASET}.yaml"

# ── Method ───────────────────────────────────────────────────────────────
METHOD = "hmlc-ssl-graph-byol"           # see table above

# ── Training ─────────────────────────────────────────────────────────────
FRACTION_LABELED = 0.1                   # fraction of labeled data (0.01 to 1.0)
SEED = 42
EPOCHS = 20
BATCH_SIZE = 16
LR = 1e-4
MAX_LR = 3e-4
PATIENCE = 5
PRETRAINED = True                        # use ImageNet-pretrained ViT weights
DEVICES = [0]                            # GPU IDs

# ── Mode control ─────────────────────────────────────────────────────────
SKIP_TRAINING = True                     # True = load model, False = train
SKIP_INFERENCE = False                   # True = skip prediction

# ── Method config lookup ─────────────────────────────────────────────────
import yaml
with open(f"configs/method/{METHOD}.yaml") as f:
    method_cfg = yaml.safe_load(f)

LEARNING_TASK = method_cfg["learning_task"]   # mlc or hmlc
TRAINING_MODE = method_cfg["training_mode"]   # sl or ssl
APPLY_EDGES = method_cfg["apply_edges"]
APPLY_BYOL = method_cfg["apply_byol"]

print(f"Dataset:        {DATASET}")
print(f"Method:         {METHOD}")
print(f"Learning task:  {LEARNING_TASK}")
print(f"Training mode:  {TRAINING_MODE} ({'semi-supervised' if TRAINING_MODE == 'ssl' else 'supervised'})")
print(f"Graph (SAGE):   {APPLY_EDGES}")
print(f"BYOL:           {APPLY_BYOL}")
print(f"Fraction:       {int(FRACTION_LABELED * 100)}%")
print(f"Epochs:         {EPOCHS}")
print(f"Batch size:     {BATCH_SIZE}")

## 3. Prepare data pipeline

Downloads the dataset from HuggingFace on first run and caches as `.npy` files. Subsequent runs load from cache.

In [ ]:
seed_everything(SEED, workers=True)

# Load hierarchy + dataset
pipeline = DatasetPipeline(
    yaml_path=DATASET_CONFIG,
    seed=SEED,
    cache_dir="./Datasets/mlc_datasets_npy",
)

fraction = FRACTION_LABELED if TRAINING_MODE == "ssl" else None
outputs = pipeline.run_pipeline(fraction_labeled=fraction)

# In supervised mode, remove unlabeled data
if TRAINING_MODE == "sl":
    outputs.pop("U", None)

# Resolve dimensions
num_leaves = pipeline.num_classes
num_classes = num_leaves if LEARNING_TASK == "mlc" else pipeline.num_classes_extended

# Edge index for graph methods
edge_index = None
if APPLY_EDGES and LEARNING_TASK == "hmlc":
    edge_index = create_edge_index(hierarchy=pipeline.label_to_predecessors)

print(f"\nClasses: {num_classes} total ({num_leaves} leaves)")
print(f"Train:   {len(outputs['X'][0])} samples")
print(f"Val:     {len(outputs['X_val'])} samples")
print(f"Test:    {len(outputs['X_te'])} samples")
if "U" in outputs:
    print(f"Unlabel: {len(outputs['U'])} samples")
if edge_index is not None:
    print(f"Edges:   {edge_index.shape[1]} hierarchy edges")

In [ ]:
# Create datamodule
transforms = Preprocess()
datamodule = BaseDataModule(
    outputs,
    batch_size=BATCH_SIZE,
    num_workers=2,
    transforms=transforms,
)

# Build config dict for trainer
config = Dotdict({
    "training": {
        "lr": LR, "head_lr": LR, "max_lr": MAX_LR,
        "apply_scheduler": True, "epochs": EPOCHS,
        "min_epochs": 1, "patience": PATIENCE,
        "lr_schedule_patience": 5,
        "accumulate_grad_batches": 5,
        "deterministic": True, "log_every_n_steps": 1,
    },
    "dataset": {
        "name": DATASET, "folder_name": DATASET,
        "num_classes": num_leaves,
    },
})

print("DataModule and config ready.")

## 4. Train (optional)

Set `SKIP_TRAINING = False` in the configuration cell to train from scratch.
Otherwise, skip to section 5 to load a pre-trained model.

In [ ]:
if not SKIP_TRAINING:
    # Build model
    backbone = h_deit_base_embedding(num_classes=num_classes, pretrained=PRETRAINED)
    trainer_cls = get_trainer_class(TRAINING_MODE, LEARNING_TASK, APPLY_EDGES, APPLY_BYOL)
    lightning_model = trainer_cls(config, backbone, num_leaves, LEARNING_TASK, edge_index)

    print(f"Model: {trainer_cls.__name__}")
    total_params = sum(p.numel() for p in lightning_model.parameters())
    print(f"Parameters: {total_params / 1e6:.1f}M")

    # Callbacks
    frac_int = int(100 * FRACTION_LABELED)
    save_dir = f"saved_models/{DATASET}/{METHOD}/fraction_{frac_int}/seed_{SEED}"
    callbacks = [
        RichProgressBar_(),
        EarlyStopping_(metric="val_loss", mode="min", patience=PATIENCE),
        ModelCheckpoint_(dirpath=save_dir, metric="val_loss", mode="min"),
    ]

    # Train
    trainer = L.Trainer(
        accelerator="auto",
        devices=DEVICES,
        strategy="auto",
        precision="16-mixed",
        max_epochs=EPOCHS,
        min_epochs=1,
        accumulate_grad_batches=5,
        enable_checkpointing=True,
        enable_model_summary=True,
        num_sanity_val_steps=0,
        log_every_n_steps=1,
        callbacks=callbacks,
    )
    trainer.fit(lightning_model, datamodule=datamodule)

    # Save final model weights
    os.makedirs(save_dir, exist_ok=True)
    torch.save(lightning_model.state_dict(), os.path.join(save_dir, "final_model.pt"))
    print(f"\nModel saved to {save_dir}/final_model.pt")
else:
    print("Skipping training (SKIP_TRAINING = True). Change to False to train.")

## 5. Load model for inference

Load a trained model — either from training above, from a local checkpoint, or downloaded from Google Drive.

In [ ]:
# ── Option A: Load from training above ───────────────────────────────────
# (already in memory as `lightning_model` if SKIP_TRAINING was False)

# ── Option B: Load from saved weights ───────────────────────────────────
frac_int = int(100 * FRACTION_LABELED)
model_dir = f"saved_models/{DATASET}/{METHOD}/fraction_{frac_int}/seed_{SEED}"
model_path = os.path.join(model_dir, "final_model.pt")

# ── Option C: Download from Google Drive (uncomment and set file ID) ────
# GDRIVE_FILE_ID = "YOUR_GOOGLE_DRIVE_FILE_ID"
# !pip install -q gdown
# import gdown
# os.makedirs(model_dir, exist_ok=True)
# gdown.download(id=GDRIVE_FILE_ID, output=model_path, quiet=False)

# Build model and load weights
backbone = h_deit_base_embedding(num_classes=num_classes, pretrained=False)
trainer_cls = get_trainer_class(TRAINING_MODE, LEARNING_TASK, APPLY_EDGES, APPLY_BYOL)
lightning_model = trainer_cls(config, backbone, num_leaves, LEARNING_TASK, edge_index)

if os.path.exists(model_path):
    state = torch.load(model_path, map_location="cpu")
    lightning_model.load_state_dict(state)
    print(f"Loaded weights from {model_path}")
elif not SKIP_TRAINING:
    print("Using model from training (already in memory).")
else:
    # Try loading from checkpoint (.ckpt)
    ckpt_files = glob.glob(os.path.join(model_dir, "*.ckpt"))
    if ckpt_files:
        ckpt_path = ckpt_files[0]
        lightning_model = trainer_cls.load_from_checkpoint(
            ckpt_path, config=config, backbone=backbone,
            num_leaves=num_leaves, learning_task=LEARNING_TASK, edge_index=edge_index,
        )
        print(f"Loaded checkpoint from {ckpt_path}")
    else:
        print(f"No model found at {model_dir}/")
        print("Set SKIP_TRAINING = False to train, or provide a model path.")

lightning_model.eval()
print("Model ready for inference.")

## 6. Run inference and evaluate

In [ ]:
if not SKIP_INFERENCE:
    # Run prediction on test set
    inference_trainer = L.Trainer(
        accelerator="auto",
        devices=DEVICES,
        strategy="auto",
        enable_model_summary=False,
    )
    Y = predict(inference_trainer, lightning_model, datamodule)

    # Calculate metrics
    df_metrics = calculate_metrics(Y)
    df_metrics.columns = ["Value"]
    print("\n" + "=" * 45)
    print(f"  {METHOD} on {DATASET} ({int(FRACTION_LABELED*100)}% labeled)")
    print("=" * 45)
    display(df_metrics.style.format({"Value": "{:.4f}"}))

    # Save metrics
    results_dir = f"saved_models/{DATASET}/{METHOD}/fraction_{frac_int}/seed_{SEED}/results"
    os.makedirs(results_dir, exist_ok=True)
    df_metrics.to_csv(os.path.join(results_dir, "metrics.txt"), sep="\t")
    print(f"\nMetrics saved to {results_dir}/metrics.txt")
else:
    print("Skipping inference (SKIP_INFERENCE = True).")

## 7. Visualize predictions

Shows random test images with predicted vs true labels. Re-run this cell for different samples.

In [ ]:
if not SKIP_INFERENCE:
    # Get leaf label names from config
    with open(DATASET_CONFIG) as f:
        ds_cfg = yaml.safe_load(f)
    CLASS_NAMES = ds_cfg["leaf_labels"]

    N_SHOW = 4
    n_test = Y["y_true"].shape[0]
    indices = np.random.choice(n_test, N_SHOW, replace=False)

    fig, axes = plt.subplots(2, N_SHOW, figsize=(4 * N_SHOW, 7),
                              gridspec_kw={"height_ratios": [1, 1.2]})

    for col, idx in enumerate(indices):
        # ── Row 0: image ──
        ax_img = axes[0, col]
        # Get raw image from test set
        img = outputs["X_te"][idx]
        if img.max() > 1:
            img = img.astype(np.float32) / 255.0
        img = np.moveaxis(img, 0, -1)  # CHW -> HWC
        ax_img.imshow(np.clip(img, 0, 1))
        ax_img.set_xticks([])
        ax_img.set_yticks([])

        # True labels
        true_idx = np.where(Y["y_true"][idx] > 0)[0]
        true_names = [CLASS_NAMES[i] for i in true_idx if i < len(CLASS_NAMES)]
        ax_img.set_title(", ".join(true_names) or "none", fontsize=9, wrap=True)

        # ── Row 1: prediction bar chart ──
        ax_bar = axes[1, col]
        scores = Y["y_scores"][idx][:len(CLASS_NAMES)]
        top_k = np.argsort(scores)[-5:][::-1]
        top_names = [CLASS_NAMES[k] for k in top_k]
        top_vals = [scores[k] for k in top_k]
        bar_colors = ["#2E86AB" if Y["y_true"][idx][k] > 0 else "#ccc" for k in top_k]

        ax_bar.barh(range(len(top_k)), top_vals, color=bar_colors)
        ax_bar.set_yticks(range(len(top_k)))
        ax_bar.set_yticklabels(top_names, fontsize=9)
        ax_bar.set_xlim(0, 1)
        ax_bar.invert_yaxis()
        if col == 0:
            ax_bar.set_ylabel("Top-5 predictions", fontsize=10)

    plt.suptitle(
        f"{METHOD} predictions on {DATASET} ({int(FRACTION_LABELED*100)}% labeled)\n"
        "Blue = correct, Gray = incorrect",
        fontsize=12, y=1.02,
    )
    plt.tight_layout()
    plt.show()
else:
    print("Run inference first (set SKIP_INFERENCE = False).")

## 8. Compare methods

Run the same dataset with different method variants and compare metrics side-by-side.

In [ ]:
# Load saved metrics from multiple methods (if available)
COMPARE_METHODS = [
    "mlc-sl",
    "hmlc-sl",
    "hmlc-sl-graph",
    "hmlc-ssl-graph",
    "hmlc-ssl-byol",
    "hmlc-ssl-graph-byol",
]

METRICS_TO_SHOW = ["micro f1", "macro f1", "average auprc", "ranking loss", "subset accuracy", "hamming loss"]

METHOD_LABELS = {
    "mlc-sl": "MLC (flat)",
    "hmlc-sl": "HMLC",
    "hmlc-sl-graph": "HMLC+Graph",
    "hmlc-ssl-graph": "HMLC+Graph (SSL)",
    "hmlc-ssl-byol": "HMLC+BYOL (SSL)",
    "hmlc-ssl-graph-byol": "HELM (full)",
}

METHOD_COLORS = {
    "mlc-sl": "#aaa",
    "hmlc-sl": "#E8871E",
    "hmlc-sl-graph": "#57A773",
    "hmlc-ssl-graph": "#2176AE",
    "hmlc-ssl-byol": "#9B59B6",
    "hmlc-ssl-graph-byol": "#E74C3C",
}

rows = []
for method in COMPARE_METHODS:
    p = f"saved_models/{DATASET}/{method}/fraction_{frac_int}/seed_{SEED}/results/metrics.txt"
    if os.path.exists(p):
        df = pd.read_csv(p, sep="\t", index_col=0)
        row = {"Method": METHOD_LABELS.get(method, method)}
        for m in METRICS_TO_SHOW:
            if m in df.index:
                row[m] = df.loc[m].values[0]
        rows.append(row)

if rows:
    comparison = pd.DataFrame(rows).set_index("Method")
    display(comparison.style.format("{:.4f}").highlight_max(axis=0, color="#d4edda"))

    # Bar plot
    fig, axes = plt.subplots(1, len(METRICS_TO_SHOW), figsize=(3.5 * len(METRICS_TO_SHOW), 4))
    for i, metric in enumerate(METRICS_TO_SHOW):
        ax = axes[i]
        if metric in comparison.columns:
            vals = comparison[metric]
            colors = [METHOD_COLORS.get(m, "#666") for m in COMPARE_METHODS if METHOD_LABELS.get(m, m) in vals.index]
            vals.plot.bar(ax=ax, color=colors, edgecolor="white", width=0.7)
            ax.set_title(metric, fontsize=10)
            ax.set_xlabel("")
            ax.tick_params(axis="x", rotation=45, labelsize=8)
    plt.suptitle(f"Method comparison — {DATASET} ({int(FRACTION_LABELED*100)}% labeled)", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No saved results found. Train multiple methods first, then re-run this cell.")